# Contexto do Problema e Dicionario de Dados

Instituicoes financeiras precisam estimar o risco de credito antes de aprovar uma solicitacao. Este projeto usa `data/dataset.csv` para classificar cada solicitante como **Baixo**, **Medio** ou **Alto** risco. A predicao apoia a triagem e a analise humana, mas nao substitui a decisao final.

| Nome da coluna | Tipo | Significado pratico |
|---|---|---|
| idade | int | Idade do solicitante em anos |
| renda_mensal | float | Renda mensal em reais |
| tempo_emprego_meses | int | Tempo de emprego em meses |
| score_credito | int | Pontuacao historica de credito |
| possui_imovel | str | Indicador de propriedade de imovel |
| possui_veiculo | str | Indicador de propriedade de veiculo |
| estado_civil | str | Estado civil declarado |
| nivel_escolaridade | str | Maior nivel educacional alcancado |
| indice_divida | float | Percentual aproximado da renda comprometida |
| categoria_risco | str | Variavel alvo: Baixo, Medio ou Alto |

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_csv(raiz / "data" / "dataset.csv")
df.head()

# Analise Exploratória Resumida

In [ ]:
print("Formato:", df.shape)
print("\nValores ausentes/nulos:")
print(df.isna().sum())
print("\nContagem da variavel alvo:")
print(df["categoria_risco"].value_counts())

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df["renda_mensal"], bins=30, ax=axes[0])
axes[0].set_title("Distribuicao da renda mensal")
sns.boxplot(data=df, x="categoria_risco", y="score_credito", ax=axes[1])
axes[1].set_title("Score por categoria")
sns.countplot(data=df, x="categoria_risco", ax=axes[2])
axes[2].set_title("Balanceamento do alvo")
plt.tight_layout()
plt.show()

# Construcao do Pipeline Scikit-Learn

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

X = df.drop(columns="categoria_risco")
y = df["categoria_risco"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=77, stratify=y
)

colunas_numericas = X.select_dtypes(include=np.number).columns.tolist()
colunas_categoricas = X.select_dtypes(exclude=np.number).columns.tolist()

preprocessador = ColumnTransformer(
    transformers=[
        ("numericas", StandardScaler(), colunas_numericas),
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            colunas_categoricas,
        ),
    ]
)
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    learning_rate_init=0.001,
    max_iter=500,
    random_state=77,
    early_stopping=True,
)
pipeline = Pipeline([
    ("preprocessamento", preprocessador),
    ("mlp", mlp),
])
pipeline.fit(X_treino, y_treino)

# Avaliacao do Modelo

In [ ]:
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay

y_pred = pipeline.predict(X_teste)
print(f"Acuracia global: {accuracy_score(y_teste, y_pred):.4f}")
print(f"Matthews correlation coefficient: {matthews_corrcoef(y_teste, y_pred):.4f}")

In [ ]:
rotulos = sorted(y.unique())
matriz = confusion_matrix(y_teste, y_pred, labels=rotulos)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(matriz, display_labels=rotulos).plot(ax=ax, cmap="Blues")
ax.set_title("Matriz de confusao")
plt.tight_layout()
plt.show()

# Exportacao do Modelo Treinado

In [ ]:
import joblib

(raiz / "models").mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, raiz / "models" / "modelo_mlp.joblib")
print("Modelo exportado para models/modelo_mlp.joblib")